# Step 1 — Data Loading, EDA & Feature Engineering

**Project:** Loan Decision AI Audit & Reliability Framework  
**Dataset:** Home Credit Default Risk (Kaggle)

In [ ]:
%load_ext autoreload
%autoreload 2

# Make the src/ folder importable when the notebook lives in /notebooks
import sys
...
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Standard data-science stack
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

# Our project helpers (defined in src)
from src.utils.config import load_config, get_paths
from src.utils.logging import get_logger
from src.data.loader import (
    load_application_train,
    summarise_missingness,
    dtype_inventory,
)
from src.features.build_features import (
    engineer_domain_features,
    split_feature_types,
    build_preprocessor,
)

# Load config once — every random seed, file path, and threshold lives in config.yaml
cfg = load_config()
paths = get_paths(cfg)
SEED = cfg['project']['random_seed']
TARGET = cfg['project']['target_col']
ID_COL = cfg['project']['id_col']
np.random.seed(SEED)

# Plot style — clean, no clutter
sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.dpi'] = 100

log = get_logger('step1')
log.info('Step 1 environment ready. Project root: %s', PROJECT_ROOT)

In [ ]:
df = load_application_train()
df.shape

In [ ]:
dtype_inventory(df)

In [ ]:
# Class balancing
target_share = df[TARGET].value_counts(normalize=True).rename({0: 'repaid', 1: 'defaulted'})
default_rate = df[TARGET].mean()
print(target_share.round(4))
print(f'\nDefault rate: {default_rate:.2%}')

# Visualizing it
fig, ax = plt.subplots(figsize=(5, 3))
target_share.plot(kind='bar', color=['#4C72B0', '#C44E52'], ax=ax)
ax.set_title(f'Target balance (default rate = {default_rate:.2%})')
ax.set_ylabel('share of applicants')
ax.set_xlabel('')
for spine in ('top', 'right'):
    ax.spines[spine].set_visible(False)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Top 20 columns ranked by missing fraction.
miss_top = summarise_missingness(df, top_n=20)
miss_top

In [ ]:
# How many columns cross the configured "high missingness" threshold?
thresh = cfg['eda']['high_missing_threshold']
n_high_missing = (df.isna().mean() > thresh).sum()
print(f'Columns with > {thresh:.0%} missing: {n_high_missing} / {df.shape[1]}')

# Horizontal bar chart of the worst 20, with a threshold reference line.
fig, ax = plt.subplots(figsize=(8, 5))
miss_top['missing_frac'].sort_values().plot(kind='barh', color='#55A868', ax=ax)
ax.axvline(thresh, ls='--', color='red', lw=1, label=f'{thresh:.0%} threshold')
ax.set_xlim(0, 1)
ax.set_xlabel('fraction missing')
ax.set_title('Top-20 most missing columns')
ax.legend()
for spine in ('top', 'right'):
    ax.spines[spine].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
# Nine headline numeric features a credit officer would glance at first.
key_num = ['AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY', 'AMT_GOODS_PRICE',
           'DAYS_BIRTH', 'DAYS_EMPLOYED', 'EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']
key_num = [c for c in key_num if c in df.columns]

fig, axes = plt.subplots(3, 3, figsize=(12, 8))
for ax, col in zip(axes.ravel(), key_num):
    series = df[col].dropna()
    # AMT_* are long-tailed — clip the top 1% for display readability only.
    # The underlying data is untouched.
    if col.startswith('AMT_'):
        series = series.clip(upper=series.quantile(0.99))
    ax.hist(series, bins=40, color='#4C72B0', alpha=0.85)
    ax.set_title(col, fontsize=9)
    ax.tick_params(labelsize=8)
    for spine in ('top', 'right'):
        ax.spines[spine].set_visible(False)
fig.suptitle('Headline applicant numerics (AMT_* clipped at p99 for display)', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# How well do the strongest predictors separate the two classes?
ext_cols = [c for c in ['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3'] if c in df.columns]

fig, axes = plt.subplots(1, len(ext_cols), figsize=(4 * len(ext_cols), 3.5), sharey=True)
for ax, col in zip(axes, ext_cols):
    for label, sub in df.groupby(TARGET)[col]:
        ax.hist(sub.dropna(), bins=40, alpha=0.55, density=True,
                label=('repaid' if label == 0 else 'defaulted'))
    ax.set_title(col)
    ax.legend(fontsize=8)
    for spine in ('top', 'right'):
        ax.spines[spine].set_visible(False)
fig.suptitle('External credit scores by repayment outcome', y=1.04)
plt.tight_layout()
plt.show()

In [ ]:
# How many unique values does each categorical column have?
cat_cols = df.select_dtypes(include=['object', 'str']).columns.tolist()
cardinality = df[cat_cols].nunique().sort_values()
cardinality.to_frame('n_unique')

In [ ]:
# Default rate by category for three low-cardinality, business-meaningful columns.
# Each printout shows the rate AND the sample size
for col in ('NAME_CONTRACT_TYPE', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS'):
    if col not in df.columns:
        continue
    rate = (df.groupby(col)[TARGET]
              .agg(['mean', 'count'])
              .rename(columns={'mean': 'default_rate', 'count': 'n'})
              .sort_values('default_rate', ascending=False))
    print(f'\n{col}:')
    print(rate.round(4))

In [ ]:
# Sanity-check the loader did its job. Assertion checks the exact thing we
# care about — the literal sentinel value is absent — rather than relying
# on a proxy condition like "max < 0" (the dataset legitimately contains
# DAYS_EMPLOYED == 0 for applicants who started their job on the
# application date).
SENTINEL = cfg['sentinels']['days_employed_anomaly']
assert (df['DAYS_EMPLOYED'] != SENTINEL).all(), \
    f'Sentinel {SENTINEL} should have been replaced with NaN by the loader'

# Does the anomaly flag carry signal? Compare default rates with vs without.
anom_share = df['DAYS_EMPLOYED_ANOM'].mean()
default_by_anom = df.groupby('DAYS_EMPLOYED_ANOM')[TARGET].mean()
print(f'Share of rows with DAYS_EMPLOYED sentinel: {anom_share:.2%}')
print('\nDefault rate by anomaly flag:')
print(default_by_anom.round(4))
# NOTE: my original assertion would have silently passed if I'd been working on a dataset where every applicant had been employed for at least 1 day. 
# The bug would only surface on the edge case (DAYS_EMPLOYED == 0). 

In [ ]:
# Add credit-risk-meaningful derived features. Function lives in
# src/features/build_features.py — same logic will run in training and
# serving, no train/serve skew.
df_fe = engineer_domain_features(df)

# Print every column the function added so we can audit it.
new_cols = sorted(set(df_fe.columns) - set(df.columns))
print(f'Added {len(new_cols)} engineered columns:')
for c in new_cols:
    print(f'  - {c}')

In [ ]:
# Picked the five engineered features most worth eyeballing before modelling.
engineered_ratios = ['CREDIT_INCOME_RATIO', 'ANNUITY_INCOME_RATIO',
                     'CREDIT_TERM_YEARS', 'EMPLOYED_AGE_RATIO', 'EXT_SOURCE_MEAN']
engineered_ratios = [c for c in engineered_ratios if c in df_fe.columns]
df_fe[engineered_ratios].describe().round(3)

In [ ]:
# Do the engineered features actually separate the two classes?
# Fast smell-test: mean per class, sorted by absolute difference.
# Not a substitute for SHAP, just a sanity check before we commit to them.
comparison = (df_fe.groupby(TARGET)[engineered_ratios]
                   .mean()
                   .T
                   .rename(columns={0: 'repaid_mean', 1: 'defaulted_mean'}))
comparison['abs_diff'] = (comparison['defaulted_mean'] - comparison['repaid_mean']).abs()
comparison.sort_values('abs_diff', ascending=False).round(3)

In [ ]:
# Split features into numeric vs categorical (excluding ID and target).
numeric_cols, categorical_cols = split_feature_types(df_fe, exclude=[TARGET, ID_COL])
print(f'{len(numeric_cols)} numeric, {len(categorical_cols)} categorical features')

# Build the ColumnTransformer. Defined in src/features/build_features.py
# so the exact same preprocessor will run in training, drift monitoring,
# and the Streamlit audit dashboard.
preprocessor = build_preprocessor(numeric_cols, categorical_cols)
preprocessor

In [ ]:
# Separate features from target and ID.
X = df_fe.drop(columns=[TARGET, ID_COL])
y = df_fe[TARGET]

# Fit the preprocessor (learns medians + categorical levels) and apply it.
X_matrix = preprocessor.fit_transform(X)
feature_names = preprocessor.get_feature_names_out()

print(f'Design matrix: {X_matrix.shape[0]:,} rows x {X_matrix.shape[1]:,} features')

# Sparsity calc: support both dense numpy arrays and sparse scipy matrices.
# ColumnTransformer auto-densifies when dense columns dominate (which is
# our case — 121 numeric vs ~38 one-hot columns).
nnz = X_matrix.nnz if hasattr(X_matrix, 'nnz') else np.count_nonzero(X_matrix)
total = X_matrix.shape[0] * X_matrix.shape[1]
storage = 'sparse' if hasattr(X_matrix, 'nnz') else 'dense'
print(f'Storage: {storage}  |  Sparsity: {1 - nnz / total:.2%}')

In [ ]:
import json

# Make sure the target directories exist (no-op if they already do).
paths.data_processed.mkdir(parents=True, exist_ok=True)
paths.artifacts.mkdir(parents=True, exist_ok=True)

# 1. Engineered DataFrame — Parquet preserves dtypes and is small + fast.
engineered_path = paths.data_processed / 'application_train_engineered.parquet'
df_fe.to_parquet(engineered_path, index=False)

# 2. Fitted preprocessor — joblib is sklearn's recommended serialization.
preproc_path = paths.artifacts / 'preprocessor.joblib'
joblib.dump(preprocessor, preproc_path)

# 3. Feature names — JSON so any language (not just Python) can read them.
feature_names_path = paths.artifacts / 'feature_names.json'
with feature_names_path.open('w') as fh:
    json.dump(list(feature_names), fh, indent=2)

# Confirm each artifact was written and report its size.
for p in (engineered_path, preproc_path, feature_names_path):
    print(f'  wrote {p.relative_to(PROJECT_ROOT)}  ({p.stat().st_size / 1024**2:.2f} MB)')

## Step 1 — Summary

**Built in this notebook:**

- Reproducible, config-driven data loader (`src/data/loader.py`) with explicit sentinel handling and memory downcasting.
- EDA covering class balance, missingness, headline numeric distributions, categorical default rates, and a sentinel-handling assertion.
- Domain-feature engineering producing interpretable ratios (`CREDIT_INCOME_RATIO`, `EXT_SOURCE_MEAN`, etc.) that will translate directly into SHAP reason codes in Step 3.
- A fitted `ColumnTransformer` saved to `artifacts/preprocessor.joblib` — the single source of truth for train and serve preprocessing.

**Inputs handed to Step 2:**

- `data/processed/application_train_engineered.parquet` — engineered DataFrame
- `artifacts/preprocessor.joblib` — fitted preprocessor
- `artifacts/feature_names.json` — final column ordering for SHAP labels

**Key numbers:**

- 307,511 applicants × 138 engineered features → 203 columns after preprocessing
- Default rate: 8.07% (class imbalance dictates PR-AUC and recall@precision as eval metrics in Step 2)
- DAYS_EMPLOYED sentinel handled on 18.01% of rows; the anomaly flag correlates with a ~3.3-point lower default rate
- Memory downcast saved 47% on the raw frame
